## import

In [8]:
import numpy as np
import requests
from bs4 import BeautifulSoup
from openai import OpenAI
import os
from google.colab import userdata

### OpenAI APIの設定

In [2]:
# APIキーを設定
api_key = userdata.get("APIKey")
client = OpenAI(api_key=api_key)

### WEBページからテキスト情報を抽出

In [3]:
def get_webpage_text(url):
    """指定されたURLのWebページからテキストを抽出する関数"""
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')

    # 不要なタグを除去
    for tag in soup(['script', 'style', 'header', 'footer', 'nav']):
        tag.decompose()

    # テキストを取得して整形
    text = soup.get_text()
    lines = [line.strip() for line in text.split('\n') if line.strip()]
    text = ' '.join(lines)

    return text

# 使用例
url = "https://utokyo-ipp.github.io/1/1-0.html"
text = get_webpage_text(url)
print(f"抽出されたテキスト（一部）: {text[:200]}...")

抽出されたテキスト（一部）: 1-0. Colaboratory (Colab) の使い方 — Pythonプログラミング入門  documentation 1-0. Colaboratory (Colab) の使い方¶ Colaboratory（通称 Colab）によるノートブックの操作方法について説明します。 Colaboratoryの立ち上げ¶ ブラウザにGoogleアカウント（個人でもECCSでもどちらでもよい）でログ...


### WEBページの容量確認



In [4]:
url = "https://utokyo-ipp.github.io/appendix/1-jupyter-notebook.html"
response = requests.get(url)
html = response.text
print("HTML size:", len(html) / 1024, "KB")

HTML size: 14.4775390625 KB


### テキストをチャンクに分割

In [4]:
def split_into_chunks(text, chunk_size=1000, overlap=100):
    """テキストを指定サイズのチャンクに分割する関数"""
    chunks = []
    start = 0

    while start < len(text):
        end = min(start + chunk_size, len(text))

        # 文の途中で切れないように調整
        if end < len(text):
            # 最後の空白を見つける
            while end > start and text[end] != ' ':
                end -= 1
            if end == start:  # 空白が見つからなかった場合
                end = min(start + chunk_size, len(text))

        print(f"start={start}, end={end}, next_start={max(end - overlap, start + 1)}")  # デバッグ用

        # チャンクを追加
        chunks.append(text[start:end])

        if end == len(text):   # 末尾まで行ったら終了
            break
        # 次のチャンクの開始位置（オーバーラップあり）
        start = max(end - overlap, start + 1)

    return chunks

# 使用例
chunks = split_into_chunks(text)
print(f"チャンク数: {len(chunks)}")
print(f"最初のチャンク: {chunks[0][:100]}...")

start=0, end=984, next_start=884
start=884, end=1863, next_start=1763
start=1763, end=2757, next_start=2657
start=2657, end=3454, next_start=3354
チャンク数: 4
最初のチャンク: 1-0. Colaboratory (Colab) の使い方 — Pythonプログラミング入門  documentation 1-0. Colaboratory (Colab) の使い方¶ Cola...


### テキストのベクトル化

In [5]:
def embed_text(client,text):
    """テキストをベクトル化する関数"""

    response = client.embeddings.create(
    model="text-embedding-3-small",  # 旧adaより高性能＆安価
    input=text
    )

    return response.data[0].embedding

# チャンクをベクトル化
chunk_embeddings = []
for i, chunk in enumerate(chunks):
    print(f"チャンク {i+1}/{len(chunks)} をベクトル化中...")
    embedding = embed_text(client,chunk)
    chunk_embeddings.append({
        "text": chunk,
        "embedding": embedding
    })

チャンク 1/4 をベクトル化中...
チャンク 2/4 をベクトル化中...
チャンク 3/4 をベクトル化中...
チャンク 4/4 をベクトル化中...


### 質問文とテキストの類似度を計算

In [9]:
def cosine_similarity(vec1, vec2):
    """コサイン類似度を計算する関数"""
    dot_product = np.dot(vec1, vec2)
    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)
    return dot_product / (norm1 * norm2)

def find_relevant_chunks(question, chunk_embeddings, top_k=2):
    """質問に関連するチャンクを見つける関数"""
    # 質問をベクトル化
    question_embedding = embed_text(client,question)

    # 各チャンクとの類似度を計算
    similarities = []
    for i, chunk_data in enumerate(chunk_embeddings):
        similarity = cosine_similarity(question_embedding, chunk_data["embedding"])
        similarities.append({
            "index": i,
            "similarity": similarity,
            "text": chunk_data["text"]
        })

    # 類似度の高い順にソート
    similarities.sort(key=lambda x: x["similarity"], reverse=True)

    # 上位k個を返す
    return similarities[:top_k]

# 使用例
question = "このWebページの内容は何ですか？"
relevant_chunks = find_relevant_chunks(question, chunk_embeddings)
print(f"質問: {question}")
print(f"関連性の高いチャンク: {relevant_chunks[0]['text'][:100]}...")

質問: このWebページの内容は何ですか？
関連性の高いチャンク:  Pythonプログラミング入門 View on GitHub View PDF Navigation 1-0. Colaboratory (Colab) の使い方 Colaboratoryの立ち上げ...


### 関連するチャンクをもとに回答を生成

In [12]:
def generate_answer(question, relevant_chunks):
    """関連チャンクをもとに回答を生成する関数"""
    # プロンプトの作成
    context = "\n\n".join([chunk["text"] for chunk in relevant_chunks])
    prompt = f"""以下は、Webページから抽出された情報です：

{context}

上記の情報をもとに、次の質問に答えてください：
{question}

できるだけ与えられた情報のみを使って回答してください。情報が不足している場合は、「提供された情報からは回答できません」と答えてください。"""

    # GPTで回答を生成
    response = client.chat.completions.create(
        model="gpt-4o-mini",  # 最新推奨モデル（安価で高速）
        messages=[
            {"role": "system", "content": "あなたは与えられた情報をもとに質問に答えるアシスタントです。"},
            {"role": "user", "content": prompt}
        ],
        temperature=0.3
    )

    return response.choices[0].message.content

# 使用例
answer = generate_answer(question, relevant_chunks)
print(f"回答: {answer}")

回答: このWebページは「Pythonプログラミング入門」に関する内容で、特にGoogle Colaboratory（Colab）の使い方やPythonの基本的なプログラミング概念について説明しています。具体的には、Colaboratoryの立ち上げ方、ノートブックの操作、セルの編集や挿入、数値演算、変数と関数の基礎、条件分岐、テストとデバッグ、文字列、リスト、辞書、繰り返し、ファイル入出力、モジュールの使い方、NumPyやpandas、scikit-learnライブラリの利用など、幅広いトピックが含まれています。また、ノートブックの共有やダウンロード方法についても触れられています。


### パイプラインの作成

In [15]:
def rag_qa(url, question):
    """RAGを使った質問応答システムのメイン関数"""
    # 1. Webページからテキスト抽出
    text = get_webpage_text(url)
    print("Webページからテキストを抽出しました")

    # 2. テキストをチャンクに分割
    chunks = split_into_chunks(text)
    print(f"テキストを{len(chunks)}個のチャンクに分割しました")

    # 3. チャンクをベクトル化
    chunk_embeddings = []
    for chunk in chunks:
        embedding = embed_text(client,chunk)
        chunk_embeddings.append({
            "text": chunk,
            "embedding": embedding
        })
    print("チャンクをベクトル化しました")

    # 4. 関連チャンクを見つける
    relevant_chunks = find_relevant_chunks(question, chunk_embeddings)
    print("質問に関連するチャンクを見つけました")

    # 5. 回答を生成
    answer = generate_answer(question, relevant_chunks)
    print("回答を生成しました")

    return answer

# 使用例
url = "https://example.com/your-page"
question = "このWebページの内容は何ですか？"
answer = rag_qa(url, question)
print(f"\n質問: {question}")
print(f"回答: {answer}")

Webページからテキストを抽出しました
start=0, end=202, next_start=102
テキストを1個のチャンクに分割しました
チャンクをベクトル化しました
質問に関連するチャンクを見つけました
回答を生成しました

質問: このWebページの内容は何ですか？
回答: このWebページは、例示的な例として使用するためのドメインに関する情報を提供しています。このドメインは、文書内での使用が許可されており、事前の調整や許可を求める必要はありません。さらに詳しい情報も提供されています。


### 実際に運用

In [16]:
url = "https://utokyo-ipp.github.io/1/1-0.html"
question = "このWebページの内容は何ですか？"
answer = rag_qa(url, question)
print(f"\n質問: {question}")
print(f"回答: {answer}")

Webページからテキストを抽出しました
start=0, end=984, next_start=884
start=884, end=1863, next_start=1763
start=1763, end=2757, next_start=2657
start=2657, end=3454, next_start=3354
テキストを4個のチャンクに分割しました
チャンクをベクトル化しました
質問に関連するチャンクを見つけました
回答を生成しました

質問: このWebページの内容は何ですか？
回答: このWebページは「Pythonプログラミング入門」に関する教材であり、特にGoogle Colaboratory（Colab）の使い方やPythonの基本的なプログラミング概念について説明しています。内容には、Colabの立ち上げ、ノートブックの操作、数値演算、変数と関数、条件分岐、テストとデバッグ、文字列、リスト、辞書、繰り返し、ファイル入出力、モジュールの使い方、NumPyやpandas、scikit-learnライブラリなど、多岐にわたるトピックが含まれています。また、セルの操作方法やノートブックの共有・ダウンロード方法についても触れられています。


In [17]:
url = "https://utokyo-ipp.github.io/1/1-0.html"
question = "セルの編集にはどんなコマンドがありますか？"
answer = rag_qa(url, question)
print(f"\n質問: {question}")
print(f"回答: {answer}")

Webページからテキストを抽出しました
start=0, end=984, next_start=884
start=884, end=1863, next_start=1763
start=1763, end=2757, next_start=2657
start=2657, end=3454, next_start=3354
テキストを4個のチャンクに分割しました
チャンクをベクトル化しました
質問に関連するチャンクを見つけました
回答を生成しました

質問: セルの編集にはどんなコマンドがありますか？
回答: セルの編集には以下のコマンドが使えます：

- Ctrl+c: コピー
- Ctrl+x: カット
- Ctrl+v: ペースト
- Ctrl+z: 元に戻す

これらのコマンドを使用することで、セルの内容を編集することができます。


In [19]:
url = "https://kimidori-organic.com/"
question = "どこにあるお店で、どんなことができますか？"
answer = rag_qa(url, question)
print(f"\n質問: {question}")
print(f"回答: {answer}")

Webページからテキストを抽出しました
start=0, end=613, next_start=513
テキストを1個のチャンクに分割しました
チャンクをベクトル化しました
質問に関連するチャンクを見つけました
回答を生成しました

質問: どこにあるお店で、どんなことができますか？
回答: 提供された情報からは回答できません。
